In [4]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
import numpy as np
import torch

# add parent dir so importing top level files works in notebook subdir
parent_dir = str(Path().resolve().parent)
sys.path.insert(0, parent_dir)

from llm import qwen_utils, tools

# Data paths
DATA_ROOT = Path("data/preprocessed/qwen3")
CLIP_ID = "video01_00240"  # example clip with qwen3 features

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
# Load Qwen3 patch features for a single frame
# These are stored as (N, 16384) = [main | ds0 | ds1 | ds2] concatenated features
clip_dir = DATA_ROOT / CLIP_ID
patch_feat_dir = clip_dir / "qwen3_patch_features"
instance_feat_dir = clip_dir / "qwen3_instance_features"
image_dir = clip_dir / "images"

# Load first frame's patch features for demo
frame_idx = 0
patch_feats = np.load(patch_feat_dir / f"{frame_idx:06d}_f.npy")  # (N, 16384)
print(f"Patch features shape: {patch_feats.shape}")
print(f"Feature dim: {patch_feats.shape[-1]} = 4096 * 4 (main + 3 deepstack)")

# Also load instance features for comparison
instance_feats = np.load(instance_feat_dir / f"{frame_idx:06d}_f.npy")  # (num_instances, 16384)
print(f"Instance features shape: {instance_feats.shape}")

Patch features shape: (224, 16384)
Feature dim: 16384 = 4096 * 4 (main + 3 deepstack)
Instance features shape: (7, 16384)


In [6]:
# Define tools for the agentic loop
# Tools are Dict[name -> (callable, json_spec)]
available_tools = {
    "node_distances_through_time": (tools.node_distances_through_time, tools.spec_node_distances_through_time),
    "weather_munich": (tools.weather_munich, tools.spec_weather_munich)
}

# For demo purposes, let's use just the weather tool
demo_tools = {
    "weather_munich": (tools.weather_munich, tools.spec_weather_munich)
}

In [7]:
# Load patched Qwen3-VL model with custom feature injection support
model, processor = qwen_utils.get_patched_qwen3()
print(f"Model loaded on device: {model.device}")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Model loaded on device: cuda:0


In [8]:
# Example 1: Simple generation with vision features (no tools)
# Using the full image patch features

vision_features = [torch.from_numpy(patch_feats)]

messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": "You are a helpful medical assistant analyzing cholecystectomy surgery images."}],
    },
    {
        "role": "user",
        "content": [
            {"type": "image", "image": None},
            {"type": "text", "text": "Describe what you see in this surgical image."},
        ],
    },
]

response = qwen_utils.generate_with_vision_features(
    messages=messages,
    vision_features=vision_features,
    model=model,
    processor=processor,
    qwen_version="qwen3",
    max_tokens=256,
)
print("Response:", response)


Response: This is a close-up intraoperative surgical image, likely from a laparoscopic cholecystectomy (gallbladder removal surgery).

Here’s a breakdown of what is visible:

*   **The Gallbladder:** The central focus is the gallbladder, which appears as a pale, yellowish, pear-shaped organ. Its surface is smooth and glistening, and you can see fine blood vessels (appearing as thin red lines) running across it.
*   **Surgical Instruments:** A metallic, slender instrument, likely a laparoscopic grasper or clamp, is visible on the lower left side of the image. It is positioned near the base of the gallbladder, suggesting the surgeon is preparing to dissect or clamp the cystic duct or artery.
*   **Surrounding Anatomy:** The gallbladder is nestled against surrounding reddish-brown tissue, which is likely the liver bed and adjacent abdominal structures. The liver parenchyma is visible to the right and top of the gallbladder.
*   **Surgical Field:** The image is taken from a laparoscopic pe

In [9]:
# Example 2: Agentic generation with tools
# The model can call tools and receive results in a loop

messages_with_tools = [
    {
        "role": "system",
        "content": [{"type": "text", "text": "You are a helpful medical assistant. You can use tools to get additional information. When you need weather information, use the weather_munich tool."}],
    },
    {
        "role": "user",
        "content": [
            {"type": "image", "image": None},
            {"type": "text", "text": "What do you see in this image? Also, what's the weather like in Munich on 2025-01-15?"},
        ],
    },
]

final_answer, tool_history = qwen_utils.generate_with_vision_features_agentic(
    messages=messages_with_tools,
    vision_features=vision_features,
    model=model,
    processor=processor,
    tools=demo_tools,
    qwen_version="qwen3",
    max_tokens=512,
    max_iterations=5,
    verbose=True,
)

print("\n" + "="*50)
print("FINAL ANSWER:")
print(final_answer)
print("\nTOOL CALL HISTORY:")
for call in tool_history:
    print(f"  - {call['tool_name']}({call['arguments']}) -> {call['result']}")



--- Iteration 1 ---
Model response:
The image appears to be a medical or surgical view, possibly from a laparoscopic procedure. It shows internal body tissues, including what looks like a section of intestine or bowel with visible blood vessels and surrounding tissue. A surgical instrument is also visible, suggesting an ongoing or recently completed medical procedure.

Regarding the weather in Munich on 2025-01-15, I cannot provide a forecast for that date since it is in the future. Weather forecasts are typically only available for the current and upcoming days. If you need weather information for a specific date, it would be best to check a weather service closer to that date.

If you have any other questions or need further assistance, feel free to ask!

Final answer: The image appears to be a medical or surgical view, possibly from a laparoscopic procedure. It shows internal body tissues, including what looks like a section of intestine or bowel with visible blood vessels and surr

In [ ]:
# Example 3: Using instance features (like scene graph nodes)
# Each instance gets its own "image" placeholder in the prompt

# Limit to first 5 instances for demo (fewer tokens)
num_instances = min(5, instance_feats.shape[0])
instance_vision_features = [torch.from_numpy(instance_feats[i]) for i in range(num_instances)]

# Build a prompt with multiple image placeholders
objects_content = []
for i in range(num_instances):
    objects_content.extend([
        {"type": "text", "text": f"<object id='{i}'>"},
        {"type": "image", "image": None},
        {"type": "text", "text": f"</object>\n"},
    ])

messages_multi = [
    {
        "role": "system", 
        "content": [{"type": "text", "text": "You are analyzing objects detected in a surgical scene. Each <object> tag contains visual features of a detected region."}],
    },
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "<objects>\n"},
            *objects_content,
            {"type": "text", "text": "</objects>\n\nDescribe each object you see. What surgical instruments or anatomy are visible?"},
        ],
    },
]

response = qwen_utils.generate_with_vision_features(
    messages=messages_multi,
    vision_features=instance_vision_features,
    model=model,
    processor=processor,
    qwen_version="qwen3",
    max_tokens=512,
)
print("Multi-instance response:")
print(response)


In [ ]:
# Example 4: Agentic loop with node_distances tool (simulating graph queries)
# This demonstrates the pattern for when scene graphs are available

messages_graph_query = [
    {
        "role": "system", 
        "content": [{"type": "text", "text": """You are a surgical scene analysis assistant with access to a spatial scene graph.
You can see objects in the scene and query their spatial relationships over time.
Use the node_distances_through_time tool to track how objects move relative to each other."""}],
    },
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "<objects>\n"},
            *objects_content,
            {"type": "text", "text": "</objects>\n\nAnalyze these objects. If there appear to be surgical instruments, check the distance between object 0 and object 1 over time."},
        ],
    },
]

final_answer, tool_history = qwen_utils.generate_with_vision_features_agentic(
    messages=messages_graph_query,
    vision_features=instance_vision_features,
    model=model,
    processor=processor,
    tools=available_tools,  # includes node_distances_through_time
    qwen_version="qwen3",
    max_tokens=1024,
    max_iterations=5,
    verbose=True,
)

print("\n" + "="*50)
print("FINAL ANSWER:")
print(final_answer)
print("\nTOOL CALL HISTORY:")
for call in tool_history:
    print(f"  - {call['tool_name']}({call['arguments']})")
    print(f"    Result: {call['result'][:100]}..." if len(str(call['result'])) > 100 else f"    Result: {call['result']}")
